In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [11]:
spark = SparkSession.builder.appName("NYC_traffic_speed").getOrCreate()
base_path = "/home/jovyan/work"

In [12]:
dim_df = spark.read.parquet(f"{base_path}/traffic_speeds_partitioned/dim/")
dim_df.printSchema()

root
 |-- ID: long (nullable = true)
 |-- LINK_ID: long (nullable = true)
 |-- LINK_POINTS: string (nullable = true)
 |-- ENCODED_POLY_LINE: string (nullable = true)
 |-- ENCODED_POLY_LINE_LVLS: string (nullable = true)
 |-- OWNER: string (nullable = true)
 |-- TRANSCOM_ID: long (nullable = true)
 |-- BOROUGH: string (nullable = true)
 |-- LINK_NAME: string (nullable = true)



In [4]:
dim_df = dim_df.drop("LINK_ID", "ENCODED_POLY_LINE", "ENCODED_POLY_LINE_LVLS", "OWNER", "TRANSCOM_ID", "LINK_NAME")
dim_df.printSchema()

root
 |-- ID: long (nullable = true)
 |-- LINK_POINTS: string (nullable = true)
 |-- BOROUGH: string (nullable = true)



In [5]:
dim_df_parsed = dim_df.withColumn(
    "LINK_POINTS_ARRAY",
    split(
        trim(regexp_replace(
            col("LINK_POINTS"),
            r"(\d)(\d{2}\.)",
            r"$1 $2"
        )),
        r"\s+"
    )
)

dim_df_exploded = (
    dim_df_parsed
    .withColumn("coord", explode(col("LINK_POINTS_ARRAY")))
    .filter(col("coord") != "")
    .withColumn("lat", split(col("coord"), ",")[0].cast("double"))
    .withColumn("lon", split(col("coord"), ",")[1].cast("double"))
    .filter(col("lat").isNotNull() & col("lon").isNotNull())
    .withColumn("pos", monotonically_increasing_id())
)

dim_df_exploded.show()

+---+--------------------+---------+--------------------+--------------------+----------+----------+---+
| ID|         LINK_POINTS|  BOROUGH|   LINK_POINTS_ARRAY|               coord|       lat|       lon|pos|
+---+--------------------+---------+--------------------+--------------------+----------+----------+---+
|318|40.7442206,-73.77...|   Queens|[40.7442206,-73.7...|40.7442206,-73.77...|40.7442206|-73.771661|  0|
|318|40.7442206,-73.77...|   Queens|[40.7442206,-73.7...|40.7454306,-73.76907|40.7454306| -73.76907|  1|
|318|40.7442206,-73.77...|   Queens|[40.7442206,-73.7...| 40.745701,-73.76831| 40.745701| -73.76831|  2|
|318|40.7442206,-73.77...|   Queens|[40.7442206,-73.7...| 40.7465005,-73.7654|40.7465005|  -73.7654|  3|
|318|40.7442206,-73.77...|   Queens|[40.7442206,-73.7...|40.7476404,-73.76128|40.7476404| -73.76128|  4|
|318|40.7442206,-73.77...|   Queens|[40.7442206,-73.7...|40.7480905,-73.76008|40.7480905| -73.76008|  5|
|318|40.7442206,-73.77...|   Queens|[40.7442206,-73.7..

In [6]:
w = Window.partitionBy("ID").orderBy("pos")

dim_df_dist = (
    dim_df_exploded
    .withColumn("prev_lat", lag("lat").over(w))
    .withColumn("prev_lon", lag("lon").over(w))
    .withColumn("next_lat", lead("lat").over(w))
    .withColumn("next_lon", lead("lon").over(w))
    .withColumn("dist_prev", sqrt(pow(col("lat") - col("prev_lat"), 2) + pow(col("lon") - col("prev_lon"), 2)))
    .withColumn("dist_next", sqrt(pow(col("lat") - col("next_lat"), 2) + pow(col("lon") - col("next_lon"), 2)))
)

dim_df_dist.show()

+---+--------------------+---------+--------------------+--------------------+----------+----------+---+----------+----------+----------+----------+--------------------+--------------------+
| ID|         LINK_POINTS|  BOROUGH|   LINK_POINTS_ARRAY|               coord|       lat|       lon|pos|  prev_lat|  prev_lon|  next_lat|  next_lon|           dist_prev|           dist_next|
+---+--------------------+---------+--------------------+--------------------+----------+----------+---+----------+----------+----------+----------+--------------------+--------------------+
|  1|40.74047,-74.0092...|Manhattan|[40.74047,-74.009...| 40.74047,-74.009251|  40.74047|-74.009251| 12|      NULL|      NULL|  40.74137| -74.00893|                NULL|9.555317891113536E-4|
|  1|40.74047,-74.0092...|Manhattan|[40.74047,-74.009...|  40.74137,-74.00893|  40.74137| -74.00893| 13|  40.74047|-74.009251|40.7431706|-74.008591|9.555317891113536E-4|0.001832233980689...|
|  1|40.74047,-74.0092...|Manhattan|[40.74047

In [7]:
THRESHOLD = 2.0
median_dist = dim_df_dist.groupBy("ID").agg(
    percentile_approx("dist_prev", 0.5).alias("median_dist")  # 0.5 = Median
)

dim_df_clean = (
    dim_df_dist
    .join(median_dist, on="ID")
    .filter(
        (col("dist_prev").isNull() & (col("dist_next") <= col("median_dist") * THRESHOLD)) |
        (col("dist_next").isNull() & (col("dist_prev") <= col("median_dist") * THRESHOLD)) |
        ((col("dist_prev") <= col("median_dist") * THRESHOLD) & (col("dist_next") <= col("median_dist") * THRESHOLD))
    )
)

In [8]:
dim_df_result = (
    dim_df_clean
    .withColumn("coord_clean", concat_ws(",", col("lat"), col("lon")))
    .groupBy("ID")
    .agg(collect_list("coord_clean").alias("LINK_POINTS"))
    .join(
        dim_df.drop("LINK_POINTS"),
        on="ID"
    )
    .select(dim_df.columns)
)

dim_df_result.show(200)

+---+--------------------+-------------+
| ID|         LINK_POINTS|      BOROUGH|
+---+--------------------+-------------+
|  1|[40.74047,-74.009...|    Manhattan|
|  2|[40.73933,-74.010...|    Manhattan|
|  3|[40.76375,-73.999...|    Manhattan|
|  4|[40.76491,-73.998...|    Manhattan|
|106|[40.77158,-73.994...|    Manhattan|
|110|[40.5256,-74.2303...|Staten Island|
|119|[40.70631,-74.015...|    Manhattan|
|124|[40.68036,-74.004...|    Manhattan|
|126|[40.8271606,-73.8...|        Bronx|
|129|[40.8240706,-73.8...|        Bronx|
|137|[40.8242005,-73.8...|        Bronx|
|140|[40.79789,-73.919...|       Queens|
|141|[40.77474,-73.923...|       Queens|
|142|[40.83037,-73.850...|        Bronx|
|145|[40.7081105,-73.9...|    Manhattan|
|148|[40.69153,-73.999...|     Brooklyn|
|149|[40.6916,-73.9991...|    Manhattan|
|150|[40.7016405,-73.9...|    Manhattan|
|153|[40.69153,-73.999...|     Brooklyn|
|154|[40.6757405,-74.0...|     Brooklyn|
|155|[40.71772,-73.948...|     Brooklyn|
|157|[40.69158,-

In [13]:
fact_df = spark.read.parquet(f"{base_path}/traffic_speeds_partitioned/fact/")
fact_df.printSchema()

root
 |-- SPEED: float (nullable = true)
 |-- TRAVEL_TIME: integer (nullable = true)
 |-- STATUS: byte (nullable = true)
 |-- DATA_AS_OF: string (nullable = true)
 |-- ID: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [14]:
fact_df_clean = fact_df \
    .withColumn("TIMESTAMP_TMP", to_timestamp("DATA_AS_OF", "yyyy MMM dd hh:mm:ss a")) \
    .withColumn("DATE", to_date(col("TIMESTAMP_TMP"))) \
    .withColumn("TIME", date_format(col("TIMESTAMP_TMP"), "HH:mm:ss")) \
    .drop("TIMESTAMP_TMP") \
    .drop("DATA_AS_OF") \
    .drop("STATUS")

fact_df_filtered = fact_df_clean.filter((col("TRAVEL_TIME") != 0) & (col("SPEED") != 0))

fact_df_clean.printSchema()

root
 |-- SPEED: float (nullable = true)
 |-- TRAVEL_TIME: integer (nullable = true)
 |-- ID: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- DATE: date (nullable = true)
 |-- TIME: string (nullable = true)



In [15]:
out_path = f"{base_path}/traffic_speeds_partitioned_cleaned"

In [ ]:
(dim_df_result
    .write
    .option("compression", "snappy")
    .parquet(f"{out_path}/dim/"))

In [16]:
(fact_df_filtered
    .write
    .partitionBy("ID", "year", "month")
    .option("compression", "snappy")
    .parquet(f"{out_path}/fact/"))

In [17]:
spark.stop()